# **Child Well-being - POSet creation**<br/>
**University: University of Milano-Bicocca**<br/>
**Master's Degree: Data Science (A.Y. 2025/2026)**<br/>
**Course: Data Science Lab**<br/>

---

This notebook validates the mathematical and numerical equivalence of our custom Python `poset` library against the original R package `poseticDataAnalysis`.

We run strict assertions to ensure perfect alignment across:
1. POSet adjacency structures (weak dominance and cover relations)
2. Extremal elements (minimals and maximals)
3. Exact Mutual Ranking Probabilities (ExactMRP)
4. Exact Multi-Component Separation Metrics (ExactSeparation)
5. BLS Dominance Matrices (BLSDominance)
6. Fuzzy Posetic Toolbox (FuzzyInBetweenness and FuzzySeparation)
7. Dimensionality Reduction (OptimalBidimensionalEmbedding)

**Reference:**  
Fattore M., De Capitani L., Avellone A., Suardi A. (2024).  
*A fuzzy posetic toolbox for multi-criteria evaluation on ordinal data systems.*  
Annals of Operations Research. [doi:10.1007/s10479-024-06352-3](https://link.springer.com/article/10.1007/s10479-024-06352-3?utm_source=researchgate.net&utm_medium=article)

In [1]:
import sys; sys.path.insert(0, '..')
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import poset as P

## 1. Load the 2018 Expenditure POSet Dataset

In [2]:
exp_2018 = pl.read_parquet('../data/040_public_expenditure_2018.parquet')

# Build the Python poset representation
result_py = P.poset_from_polars(
    exp_2018,
    col1='REF_AREA',
    col2='TIME_PERIOD',
    higher_is_better=True,
    dominance_mode='certain_or_possible'
)

# Extract python certain poset
pos_py = result_py['poset_certain']
elements_py = result_py['elements']

print(f"Loaded {len(elements_py)} units: {elements_py}")
print(f"Python Weak Dominance relation pairs (including reflexives): {len(pos_py.order_relation())}")

Loaded 16 units: ['SWE_2018', 'LTU_2018', 'LVA_2018', 'EST_2018', 'GBR_2018', 'FRA_2018', 'LUX_2018', 'AUT_2018', 'ESP_2018', 'PRT_2018', 'ITA_2018', 'SVN_2018', 'POL_2018', 'CZE_2018', 'SVK_2018', 'HUN_2018']
Python Weak Dominance relation pairs (including reflexives): 64


## 2. Export Python Data Structures to Temporary Files

In [3]:
# Extract dominance pairs (excluding reflexive a <= a pairs)
py_relations = [
    (a, b) for a, b in pos_py.order_relation() if a != b
]

# Save base poset data
pd.DataFrame({"elements": elements_py}).to_csv("../data/tmp/tmp_elements.csv", index=False)
pd.DataFrame(py_relations, columns=["from", "to"]).to_csv("../data/tmp/tmp_relations.csv", index=False)

# Build and save 4-variable profile grid and dummy weights
bp_mock = P.BinaryVariablePOSet(["v1", "v2", "v3", "v4"])
profiles_py = bp_mock._profiles
weights_py = np.ones(len(profiles_py), dtype=float)

pd.DataFrame(profiles_py).to_csv("../data/tmp/tmp_profiles.csv", index=False, header=False)
pd.DataFrame({"weights": weights_py}).to_csv("../data/tmp/tmp_weights.csv", index=False)

print("Temporary data structures successfully written to '../data/' for cross-validation.")

Temporary data structures successfully written to '../data/' for cross-validation.


## 3. R Validation Script Creation

In [6]:
r_script_code = """# automated_validation.R
# 1. Create a local project-specific R library folder to avoid write-permission errors
local_lib <- file.path("..", "r_library")
if (!dir.exists(local_lib)) {
    dir.create(local_lib, recursive = TRUE)
}
.libPaths(c(local_lib, .libPaths()))

# 2. Auto-install the poseticDataAnalysis package locally if it is missing
if (!requireNamespace("poseticDataAnalysis", quietly = TRUE)) {
    install.packages("poseticDataAnalysis", repos = "https://cloud.r-project.org")
}
library(poseticDataAnalysis)

# Load data generated from Python
elements_df <- read.csv("../data/tmp/tmp_elements.csv")
relations_df <- read.csv("../data/tmp/tmp_relations.csv")
profiles_df <- read.csv("../data/tmp/tmp_profiles.csv", header=FALSE)
weights_df <- read.csv("../data/tmp/tmp_weights.csv")

r_elements <- as.character(elements_df$elements)
r_dom_matrix_input <- as.matrix(relations_df)

# Instantiate S4 POSet
r_poset <- POSet(elements = r_elements, dom = r_dom_matrix_input)

# Section 4: Structural Matrix & Adjacency Exports
write.csv(as.matrix(DominanceMatrix(r_poset)), "../data/tmp/r_dom_matrix.csv", row.names = FALSE)
write.csv(as.matrix(CoverMatrix(r_poset)), "../data/tmp/r_cover_matrix.csv", row.names = FALSE)

# Section 5: Extremal Elements
write.csv(data.frame(minimals=as.character(POSetMinimals(r_poset))), "../data/tmp/r_minimals.csv", row.names = FALSE)
write.csv(data.frame(maximals=as.character(POSetMaximals(r_poset))), "../data/tmp/r_maximals.csv", row.names = FALSE)

# Section 6: Exact Mutual Ranking Probabilities (MRP)
r_mrp_out <- ExactMRP(r_poset)
write.csv(as.matrix(r_mrp_out[[1]]), "../data/tmp/r_mrp_matrix.csv", row.names = FALSE)
writeLines(as.character(r_mrp_out[[2]]), "../data/tmp/r_num_le.txt")

# Section 7: Multi-Component Separation Metrics
sep_types <- c("symmetric", "asymmetricLower", "asymmetricUpper", "vertical", "horizontal")
for (s_type in sep_types) {
    r_sep_out <- ExactSeparation(r_poset, NULL, s_type)
    write.csv(as.matrix(r_sep_out[[1]]), paste0("../data/tmp/r_sep_", s_type, ".csv"), row.names = FALSE)
}

# Section 8: BLS and Fuzzy Toolbox
r_bls_matrix <- BLSDominance(r_poset)
write.csv(as.matrix(r_bls_matrix), "../data/tmp/r_bls_matrix.csv", row.names = FALSE)

r_fsep_sym <- FuzzySeparationMinMax(r_bls_matrix, "symmetric")
write.csv(as.matrix(r_fsep_sym), "../data/tmp/r_fsep_sym.csv", row.names = FALSE)

r_finb_sym <- FuzzyInBetweennessMinMax(r_bls_matrix, "symmetric")
write.csv(as.vector(r_finb_sym), "../data/tmp/r_finb_sym_flat.csv", row.names = FALSE)

# Section 9: Dimension Reduction
r_profiles <- as.matrix(profiles_df)
r_weights <- weights_df$weights
r_opt_res <- OptimalBidimensionalEmbedding(r_profiles, r_weights)

writeLines(as.character(r_opt_res$bestLossValue), "../data/tmp/r_best_loss.txt")
write.csv(data.frame(priority=as.integer(r_opt_res$bestVariablePriority)), "../data/tmp/r_best_priority.csv", row.names = FALSE)

cat("SUCCESS: All R mathematical validations computed and exported to disk!\\n")
"""

with open("../scripts/automated_validation.R", "w") as f:
    f.write(r_script_code)
print("Created 'automated_validation.R' successfully.")

Created 'automated_validation.R' successfully.


## 4. R Script Execution

In [7]:
import os
import sys
import shutil
import subprocess

def find_rscript():
    if sys.platform == "win32":
        base_r_dir = r"C:\Program Files\R"
        if os.path.exists(base_r_dir):
            r_versions = sorted([f for f in os.listdir(base_r_dir) if f.startswith("R-")])
            if r_versions:
                latest_version = r_versions[-1]
                path_64 = os.path.join(base_r_dir, latest_version, "bin", "x64", "Rscript.exe")
                if os.path.exists(path_64):
                    return path_64
                path_std = os.path.join(base_r_dir, latest_version, "bin", "Rscript.exe")
                if os.path.exists(path_std):
                    return path_std
    rscript_path = shutil.which("Rscript")
    if rscript_path:
        return rscript_path
    return None

rscript_cmd = find_rscript()

if rscript_cmd:
    print(f"Successfully located R environment dynamically at: {rscript_cmd}")
    try:
        print("Executing cross-language mathematical validation pipeline...")
        result = subprocess.run(
            [rscript_cmd, "../scripts/automated_validation.R"],
            capture_output=True,
            text=True,
            check=True
        )
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("An error occurred during calculations inside the R environment:")
        print("--- R STDOUT ---")
        print(e.stdout if e.stdout else "(Empty)")
        print("--- R STDERR ---")
        print(e.stderr if e.stderr else "(Empty)")
else:
    print("\n --- CRITICAL ERROR: R NOT FOUND ---")

Successfully located R environment dynamically at: /usr/local/bin/Rscript
Executing cross-language mathematical validation pipeline...


KeyboardInterrupt: 

## 5. Structural Matrix & Adjacency Assertions

In [8]:
# Retrieve and format matrices from R exports
r_dom_matrix = pd.read_csv("../data/tmp/r_dom_matrix.csv").to_numpy(dtype=bool)
r_cover_matrix = pd.read_csv("../data/tmp/r_cover_matrix.csv").to_numpy(dtype=bool)

# Retrieve matrices from Python
py_dom_matrix = pos_py.dominance_matrix()
py_cover_matrix = pos_py.cover_matrix()

# Perform strict structural equality assertions
assert np.array_equal(r_dom_matrix, py_dom_matrix), "Assertion Failed: Weak dominance matrix mismatch!"
assert np.array_equal(r_cover_matrix, py_cover_matrix), "Assertion Failed: Cover matrix mismatch!"

print("ASSERTION SUCCESS: Adjacency and Cover matrices are absolutely identical!")

FileNotFoundError: [Errno 2] No such file or directory: '../data/tmp/r_dom_matrix.csv'

## 6. Extremal Boundary Element Verification

In [ ]:
# Retrieve extremal bounds from R exports
r_minimals = sorted(pd.read_csv("../data/r_minimals.csv")["minimals"].astype(str).tolist())
r_maximals = sorted(pd.read_csv("../data/r_maximals.csv")["maximals"].astype(str).tolist())

# Retrieve extremal bounds from Python
py_minimals = sorted(pos_py.minimals())
py_maximals = sorted(pos_py.maximals())

assert r_minimals == py_minimals, f"Assertion Failed: Minimals mismatch! R: {r_minimals}, Py: {py_minimals}"
assert r_maximals == py_maximals, f"Assertion Failed: Maximals mismatch! R: {r_maximals}, Py: {py_maximals}"

print("ASSERTION SUCCESS: Extremal elements match perfectly.")
print(f"  Minimals (No predecessors): {py_minimals}")
print(f"  Maximals (No successors): {py_maximals}")

## 7. Exact Mutual Ranking Probabilities (MRP) Verification

In [ ]:
# Load exact MRP calculations from R exports
r_mrp_matrix = pd.read_csv("../data/r_mrp_matrix.csv").to_numpy(dtype=float)
with open("../data/r_num_le.txt", "r") as f:
    r_num_le = int(float(f.read().strip()))

# Python exact MRP execution
py_mrp_out = P.ExactMRP(pos_py)
py_mrp_matrix = py_mrp_out
py_num_le = py_mrp_out['n_extensions']

# Assertions
assert r_num_le == py_num_le, f"Assertion Failed: Linear extension count mismatch! R: {r_num_le}, Py: {py_num_le}"
np.testing.assert_allclose(r_mrp_matrix, py_mrp_matrix, rtol=1e-12, atol=1e-12)

print(f"ASSERTION SUCCESS: Exact MRP matched over {py_num_le} linear extensions!")

## 8. Exact Multi-Component Separation Metrics Verification

In [ ]:
sep_types = ["symmetric", "asymmetricLower", "asymmetricUpper", "vertical", "horizontal"]

for s_type in sep_types:
    # Query R matrix export file
    r_sep_matrix = pd.read_csv(f"../data/r_sep_{s_type}.csv").to_numpy(dtype=float)
    
    # Query Python ExactSeparation
    py_sep_out = P.ExactSeparation(pos_py, s_type)
    py_sep_matrix = py_sep_out[s_type]
    
    # Assert numerical parity
    np.testing.assert_allclose(r_sep_matrix, py_sep_matrix, rtol=1e-12, atol=1e-12)
    print(f"ASSERTION SUCCESS: Separation component '{s_type}' matches exactly.")

## 9. BLS Dominance and Fuzzy Toolbox Verification

In [ ]:
# 1. BLS Dominance Matrix Comparison
r_bls_matrix = pd.read_csv("../data/r_bls_matrix.csv").to_numpy(dtype=float)
py_bls_matrix = P.BLSDominance(pos_py)

np.testing.assert_allclose(r_bls_matrix, py_bls_matrix, rtol=1e-12, atol=1e-12)
print("ASSERTION SUCCESS: BLS Dominance matrices are identical.")

# 2. Fuzzy Separation MinMax Comparison
r_fsep_sym = pd.read_csv("../data/r_fsep_sym.csv").to_numpy(dtype=float)
py_fsep_sym = P.FuzzySeparationMinMax(py_bls_matrix, "symmetric")["symmetric"]

np.testing.assert_allclose(r_fsep_sym, py_fsep_sym, rtol=1e-12, atol=1e-12)
print("ASSERTION SUCCESS: Fuzzy Separation MinMax (symmetric) matches.")

# 3. Fuzzy In-Betweenness MinMax Comparison
# Load flattened vector from R and recreate 3D Array using Fortran ordering
N = len(elements_py)
r_finb_flat = pd.read_csv("../data/r_finb_sym_flat.csv").to_numpy(dtype=float).flatten()
r_finb_sym = r_finb_flat.reshape((N, N, N), order='F')

py_finb_sym = P.FuzzyInBetweennessMinMax(py_bls_matrix, "symmetric")["symmetric"]

np.testing.assert_allclose(r_finb_sym, py_finb_sym, rtol=1e-12, atol=1e-12)
print("ASSERTION SUCCESS: Fuzzy In-Betweenness MinMax (symmetric 3D array) matches.")

## 10. Dimension Reduction (Optimal Bidimensional Embedding) Verification

In [ ]:
# 1. Run Python Optimal Bidimensional Embedding
py_opt = P.OptimalBidimensionalEmbedding(profiles_py, weights_py)
py_best_loss = py_opt["bestLossValue"]
# Convert 0-indexed Python priority indices to 1-indexed R priority indices
py_best_priority = [x + 1 for x in py_opt["bestVariablePriority"]]

# 2. Load R results from output text variables
with open("../data/r_best_loss.txt", "r") as f:
    r_best_loss = float(f.read().strip())
r_best_priority = pd.read_csv("../data/r_best_priority.csv")["priority"].tolist()

# Assertions
assert abs(r_best_loss - py_best_loss) < 1e-12, f"Assertion Failed: Loss mismatch! R: {r_best_loss}, Py: {py_best_loss}"
assert r_best_priority == py_best_priority, f"Assertion Failed: Variable priority mismatch! R: {r_best_priority}, Py: {py_best_priority}"

print("\n--- Bidimensional Embedding Validation ---")
print(f"Optimal Loss verified: {py_best_loss:.8f}")
print(f"Optimal Variable Priority verified: {py_best_priority}")
print("ASSERTION SUCCESS: Optimal Bidimensional Embedding results match exactly.")